In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

# 1. Inisialisasi SparkSession
spark = SparkSession.builder.appName("Tugas6_ETL_Produksi").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 2. Membaca ketiga sumber data
df_trx_tugas = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk_tugas = spark.read.json("tugas6_produk.json")
df_ulasan_tugas = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

# 3. Menampilkan jumlah baris dan skema
print(f"--- Data Transaksi ({df_trx_tugas.count()} baris) ---")
df_trx_tugas.printSchema()

print(f"\n--- Data Produk ({df_produk_tugas.count()} baris) ---")
df_produk_tugas.printSchema()

print(f"\n--- Data Ulasan ({df_ulasan_tugas.count()} baris) ---")
df_ulasan_tugas.printSchema()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/27 16:05:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

--- Data Transaksi (5000 baris) ---
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)


--- Data Produk (30 baris) ---
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)


--- Data Ulasan (3500 baris) ---
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [3]:
# 1. Join Transaksi ke Ulasan
# Menggunakan LEFT JOIN karena tidak semua transaksi memiliki ulasan 
df_etl_join1 = df_trx_tugas.join(df_ulasan_tugas, on="order_id", how="left")

# 2. Join hasil di atas ke Produk
# Menggunakan INNER JOIN karena diasumsikan setiap transaksi pasti memiliki produk_id yang valid di master produk
df_etl_join2 = df_etl_join1.join(df_produk_tugas, on="product_id", how="inner")

# 3. Menambahkan kolom total_pendapatan
df_etl_transform = df_etl_join2.withColumn(
    "total_pendapatan", col("unit_terjual") * col("harga")
)

df_etl_transform.show(5)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
o

In [4]:
# 1. Menambahkan kolom ada_ulasan SEBELUM mengisi nilai null
# Jika kolom rating memiliki nilai (tidak null), maka True. Jika null, maka False.
df_etl_transform = df_etl_transform.withColumn(
    "ada_ulasan",
    when(col("rating").isNotNull(), True).otherwise(False)
)

# 2. Mengisi rating yang kosong (null) dengan angka 0
df_etl_final = df_etl_transform.fillna(0, subset=["rating"])

df_etl_final.select("order_id", "nama_produk", "kategori", "rating", "ada_ulasan").show(5)

+--------+-----------+------------+------+----------+
|order_id|nama_produk|    kategori|rating|ada_ulasan|
+--------+-----------+------------+------+----------+
|     TX0|  Produk-21|  Elektronik|     1|      true|
|     TX1|   Produk-8|     Fashion|     0|     false|
|     TX2|  Produk-25|  Elektronik|     2|      true|
|     TX3|   Produk-3|     Fashion|     5|      true|
|     TX4|  Produk-19|Rumah Tangga|     0|     false|
+--------+-----------+------------+------+----------+
only showing top 5 rows



In [5]:
# Tentukan path tujuan (Ganti 'mahasiswa' dengan username yang sesuai jika diperlukan)
output_path = "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"

# Membuat direktori parent jika belum ada (opsional)
!hdfs dfs -mkdir -p /user/mahasiswa/tugas6

# Menyimpan data dalam format Parquet dengan partisi kategori
df_etl_final.write.mode("overwrite").partitionBy("kategori").parquet(output_path)

print("Proses Load Selesai. Verifikasi struktur folder:")
!hdfs dfs -ls -R /user/mahasiswa/tugas6/hasil_etl

# Verifikasi data dengan membacanya kembali
df_verify = spark.read.parquet(output_path)
print(f"\nJumlah baris data tersimpan di HDFS: {df_verify.count()}")

Proses Load Selesai. Verifikasi struktur folder:
-rw-r--r--   3 tara supergroup          0 2026-09-27 16:06 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - tara supergroup          0 2026-09-27 16:06 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 tara supergroup      16363 2026-09-27 16:06 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik/part-00000-d50195ce-e7ed-4db4-b3a9-d983afa6e756.c000.snappy.parquet
drwxr-xr-x   - tara supergroup          0 2026-09-27 16:06 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 tara supergroup      11130 2026-09-27 16:06 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion/part-00000-d50195ce-e7ed-4db4-b3a9-d983afa6e756.c000.snappy.parquet
drwxr-xr-x   - tara supergroup          0 2026-09-27 16:06 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 tara supergroup      10388 2026-09-27 16:06 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan/part-00000-d50195ce-e7ed-4db4-b3a9-d983afa6e756.c0

In [ ]:
from pyspark.sql.functions import sum as spark_sum, count, round

# Agregasi untuk menghitung persentase ulasan per kategori
df_insight = df_verify.groupBy("kategori").agg(
    count("order_id").alias("total_transaksi"),
    spark_sum(col("ada_ulasan").cast("int")).alias("transaksi_dengan_ulasan")
).withColumn(
    "persentase_ulasan", 
    round((col("transaksi_dengan_ulasan") / col("total_transaksi")) * 100, 2)
).orderBy("persentase_ulasan")

df_insight.show()

In [ ]:
Mengetahui kategori produk dengan persentase ulasan terendah sangat penting bagi tim marketing untuk mengidentifikasi area yang kurang memiliki social proof (bukti sosial) atau interaksi pelanggan. Dengan wawasan ini, tim dapat merancang kampanye bertarget—seperti memberikan insentif poin atau diskon pasca-pembelian—untuk secara spesifik mendorong pelanggan di kategori tersebut agar bersedia meninggalkan ulasan. Peningkatan jumlah ulasan organik ini krusial karena berbanding lurus dengan meningkatnya kepercayaan calon pembeli baru, yang pada akhirnya akan mendongkrak penjualan pada kategori yang tertinggal tersebut.